In [ ]:
import pyodbc
print(pyodbc.drivers())

In [ ]:
import os, pandas as pd
from decimal import Decimal

SERVER = os.environ.get("DB_SERVER", r"localhost\SQLEXPRESS")   # apna SQL Server instance name (SSMS connect dialog wala)
CONN = (f"DRIVER={{ODBC Driver 18 for SQL Server}};SERVER={SERVER};DATABASE=school_transport_db;"
        "Trusted_Connection=yes;TrustServerCertificate=yes;")

conn = pyodbc.connect(CONN)
df = pd.read_sql("SELECT TOP 5 * FROM vw_student_transport_details", conn)
conn.close()
df

In [ ]:
import os, re, pyodbc, pandas as pd, gradio as gr, plotly.express as px
from decimal import Decimal
from datetime import date

SERVER = os.environ.get("DB_SERVER", r"localhost\SQLEXPRESS")   # apna SQL Server instance name
CONN = (f"DRIVER={{ODBC Driver 18 for SQL Server}};SERVER={SERVER};DATABASE=school_transport_db;"
        "Trusted_Connection=yes;TrustServerCertificate=yes;")

def clean(e):
    return re.sub(r"\[.*?\]|\(\d+\)|\(SQLExecDirectW\)|[()']", "", str(e)).strip(" ,")

def to_df(cur):
    cols = [d[0] for d in cur.description]
    rows = [[float(v) if isinstance(v, Decimal) else v for v in r] for r in cur.fetchall()]
    return pd.DataFrame(rows, columns=cols)

def run(sql, params=()):
    c = pyodbc.connect(CONN); cur = c.cursor()
    try:
        cur.execute(sql, params); return to_df(cur)
    finally:
        cur.close(); c.close()

def dml(sql, params):
    c = pyodbc.connect(CONN); cur = c.cursor()
    try:
        cur.execute(sql, params); c.commit(); return "✅ Saved successfully"
    except pyodbc.Error as e:
        return "❌ " + clean(e)
    finally:
        cur.close(); c.close()

def exec_proc(sql, params=()):
    c = pyodbc.connect(CONN); cur = c.cursor()
    try:
        cur.execute(sql, params)
        df = to_df(cur) if cur.description else pd.DataFrame()
        c.commit()
        return "✅ Success", df
    except pyodbc.Error as e:
        return "❌ " + clean(e), pd.DataFrame()
    finally:
        cur.close(); c.close()

# ---- 13 reports ko 04_reports.sql se load karo ----
txt = open("04_reports.sql", encoding="utf-8").read()
parts = re.split(r"^-- (Q\d+: .+)$", txt, flags=re.M)
REPORTS = {parts[i].strip(): parts[i + 1].strip().rstrip(";") for i in range(1, len(parts), 2)}

# ---- Dashboard ----
def dashboard():
    k = run("""SELECT (SELECT COUNT(*) FROM Students WHERE status='Active') AS students,
                      (SELECT COUNT(*) FROM Buses WHERE status='Active') AS buses,
                      (SELECT COUNT(*) FROM Drivers WHERE status='Active') AS drivers,
                      (SELECT COUNT(*) FROM Routes) AS routes,
                      (SELECT COALESCE(SUM(amount),0) FROM Payments WHERE status IN ('Paid','Late')) AS revenue""").iloc[0]
    md = (f"## 🎓 {int(k['students'])} Students &nbsp;|&nbsp; 🚌 {int(k['buses'])} Active Buses &nbsp;|&nbsp; "
          f"👨‍✈️ {int(k['drivers'])} Drivers &nbsp;|&nbsp; 🛣️ {int(k['routes'])} Routes &nbsp;|&nbsp; 💰 Rs {int(k['revenue']):,}")
    f1 = px.bar(run("""SELECT r.route_name, COUNT(s.student_id) AS students FROM Routes r
                       LEFT JOIN Stops st ON st.route_id=r.route_id
                       LEFT JOIN Students s ON s.stop_id=st.stop_id AND s.status='Active'
                       GROUP BY r.route_id, r.route_name"""),
                x="route_name", y="students", title="Active Students per Route")
    f2 = px.bar(run("""SELECT month_paid, SUM(amount) AS revenue FROM Payments
                       WHERE status IN ('Paid','Late') GROUP BY month_paid ORDER BY month_paid"""),
                x="month_paid", y="revenue", title="Fee Revenue by Month")
    return md, f1, f2

# ---- Students ----
def students_table(q=""):
    return run("SELECT * FROM vw_student_transport_details WHERE student_name LIKE ? ORDER BY student_id", (f"%{q}%",))

def register(name, cls, gender, guardian, phone, address, stop_id):
    if not (name and cls and gender and guardian and phone and stop_id):
        return "❌ Please fill all required fields", students_table()
    msg, _ = exec_proc("EXEC sp_register_student ?,?,?,?,?,?,?",
                       (name, cls, gender, guardian, phone, address or "", int(stop_id)))
    return msg, students_table()

stop_choices = [(f"{r.stop_name} — {r.route_name}", int(r.stop_id)) for r in run(
    "SELECT st.stop_id, st.stop_name, r.route_name FROM Stops st JOIN Routes r ON r.route_id=st.route_id ORDER BY r.route_id, st.stop_order").itertuples()]
student_choices = [(f"{r.student_id} - {r.full_name}", int(r.student_id)) for r in run(
    "SELECT student_id, full_name FROM Students WHERE status='Active'").itertuples()]

# ---- Payments ----
def pay(student_id, month, amount, method, pdate):
    if not student_id:
        return "❌ Select a student"
    return dml("INSERT INTO Payments (student_id, amount, payment_date, month_paid, method) VALUES (?,?,?,?,?)",
               (student_id, amount, pdate, month, method))

def fee_report(month):
    return exec_proc("EXEC sp_monthly_fee_report ?", (month,))[1]

# ---- Occupancy ----
def occupancy():
    df = run("SELECT * FROM vw_route_occupancy ORDER BY occupancy_pct DESC")
    return df, px.bar(df, x="route_name", y="occupancy_pct", title="Route Occupancy %")

# ---- Reports ----
def run_report(name):
    return run(REPORTS[name])

with gr.Blocks(title="School Transport System") as demo:
    gr.Markdown("# 🚌 School Transport & Bus Route Management System\nDBMS Project — SQL Server + Python + Gradio")

    with gr.Tab("📊 Dashboard"):
        kpi = gr.Markdown(); p1 = gr.Plot(); p2 = gr.Plot()
        gr.Button("Refresh").click(dashboard, outputs=[kpi, p1, p2])
        demo.load(dashboard, outputs=[kpi, p1, p2])

    with gr.Tab("🎓 Students"):
        q = gr.Textbox(label="Search by name")
        tbl = gr.Dataframe(value=students_table(), interactive=False)
        q.change(students_table, q, tbl)
        gr.Markdown("### Register New Student (stored procedure `sp_register_student`)")
        with gr.Row():
            n = gr.Textbox(label="Full name"); c = gr.Textbox(label="Class (e.g. Class 5)")
            g = gr.Dropdown(["Male", "Female"], label="Gender")
        with gr.Row():
            gn = gr.Textbox(label="Guardian name"); gp = gr.Textbox(label="Guardian phone")
        ad = gr.Textbox(label="Address"); sp = gr.Dropdown(stop_choices, label="Pickup stop")
        out = gr.Textbox(label="Result")
        gr.Button("Register", variant="primary").click(register, [n, c, g, gn, gp, ad, sp], [out, tbl])

    with gr.Tab("💰 Payments"):
        gr.Markdown("### Record Payment (trigger `trg_payment_late` marks payments after the 10th as Late)")
        with gr.Row():
            ps = gr.Dropdown(student_choices, label="Student")
            pm = gr.Textbox(label="Month (YYYY-MM)", value="2026-11")
            pa = gr.Number(label="Amount", value=3500)
        with gr.Row():
            pme = gr.Dropdown(["Cash", "Bank Transfer", "JazzCash", "Easypaisa"], value="Cash", label="Method")
            pd_ = gr.Textbox(label="Payment date (YYYY-MM-DD)", value=str(date.today()))
        pout = gr.Textbox(label="Result")
        gr.Button("Save Payment", variant="primary").click(pay, [ps, pm, pa, pme, pd_], pout)
        gr.Markdown("### Monthly Fee Report (stored procedure `sp_monthly_fee_report`)")
        fm = gr.Textbox(label="Month", value="2026-10"); ft = gr.Dataframe(interactive=False)
        gr.Button("Generate").click(fee_report, fm, ft)

    with gr.Tab("🛣️ Route Occupancy"):
        ot = gr.Dataframe(interactive=False); of = gr.Plot()
        gr.Button("Load").click(occupancy, outputs=[ot, of])

    with gr.Tab("📑 SQL Reports"):
        dd = gr.Dropdown(list(REPORTS.keys()), label="Choose report", value=list(REPORTS.keys())[0])
        rt = gr.Dataframe(interactive=False)
        gr.Button("Run Report", variant="primary").click(run_report, dd, rt)
        dd.change(run_report, dd, rt)

# Optional login + public link: set APP_USER and APP_PASS environment variables first.
APP_USER = os.environ.get("APP_USER")
APP_PASS = os.environ.get("APP_PASS")
if APP_USER and APP_PASS:
    demo.launch(inline=False, share=True, auth=(APP_USER, APP_PASS), theme=gr.themes.Soft())
else:
    demo.launch(inline=False, theme=gr.themes.Soft())